# SimCLR on CIFAR-10

这个 Notebook 展示如何在 `CIFAR-10` 上实现一个完整的 `SimCLR` 流程，并重点解释对比学习里的 `NT-Xent / InfoNCE loss` 是怎么计算的。

内容包括：
- 双视图数据增强
- encoder + projection head
- `NT-Xent` 损失函数实现
- 对比预训练流程
- 特征提取
- 线性分类评估示例
- `SimCLR loss` 的详细计算逻辑

## 1. 环境准备

如果本地环境尚未安装依赖，可以先执行：

```bash
pip install torch torchvision matplotlib
```

In [ ]:
# dataclass 用于集中管理实验配置
from dataclasses import dataclass

# matplotlib 用于样本和训练曲线可视化
import matplotlib.pyplot as plt
# PyTorch 核心模块
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
# torchvision 提供数据集、增强和 backbone
from torchvision import datasets, models, transforms

# 设置绘图风格，便于 Notebook 展示
plt.style.use('seaborn-v0_8')
# 固定随机种子，便于复现
torch.manual_seed(42)

# 优先使用 GPU，没有则退回 CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    # 数据集下载与缓存目录
    data_root: str = './data'
    # 将 CIFAR-10 统一拉伸到 224x224，便于和前面几个 Notebook 保持一致
    image_size: int = 224
    # 对比学习预训练 batch size
    batch_size: int = 64
    # DataLoader 并行加载进程数
    num_workers: int = 2
    # 对比学习学习率
    lr: float = 1e-3
    # 对比学习训练轮数
    epochs: int = 5
    # 温度系数，控制 softmax 的平滑程度
    temperature: float = 0.5
    # projection head 输出维度
    projection_dim: int = 128
    # 线性评估阶段 batch size
    linear_batch_size: int = 128
    # 线性评估阶段训练轮数
    linear_epochs: int = 3


cfg = Config()
cfg

## 2. SimCLR 的核心思路

SimCLR 的关键不是直接做分类，而是先学出一个好的特征空间：

1. 对同一张图片做两次随机增强，得到两个不同视图。
2. 把这两个视图送进同一个 encoder。
3. 再通过 projection head 映射到对比学习空间。
4. 希望同一张图片的两个视图更接近，不同图片的视图更远离。

因此，SimCLR 的监督信号并不是类别标签，而是“这两个样本是不是来自同一张原图”。

In [ ]:
# CIFAR-10 常用归一化参数
cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2470, 0.2435, 0.2616)

# SimCLR 的关键之一是比较强的数据增强
simclr_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.RandomResizedCrop(cfg.image_size, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)
    ], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

# 线性评估阶段不需要双视图增强，只保留较常规的预处理
eval_train_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

eval_test_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

In [ ]:
class SimCLRPairDataset(Dataset):
    def __init__(self, root, train=True, transform=None, download=True):
        # 底层仍然使用 CIFAR-10 数据集
        self.dataset = datasets.CIFAR10(root=root, train=train, download=download)
        self.transform = transform
        self.classes = self.dataset.classes

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # 取出原始 PIL 图像和标签
        image, label = self.dataset[idx]
        # 对同一张图片做两次独立随机增强，得到两个视图
        view1 = self.transform(image)
        view2 = self.transform(image)
        return view1, view2, label


contrastive_dataset = SimCLRPairDataset(
    root=cfg.data_root,
    train=True,
    transform=simclr_transform,
    download=True,
)

linear_train_dataset = datasets.CIFAR10(
    root=cfg.data_root,
    train=True,
    download=True,
    transform=eval_train_transform,
)

linear_test_dataset = datasets.CIFAR10(
    root=cfg.data_root,
    train=False,
    download=True,
    transform=eval_test_transform,
)

classes = contrastive_dataset.classes
classes

In [ ]:
def denormalize(image_tensor, mean, std):
    # 反归一化，便于把图像恢复到可视化范围
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return image_tensor * std + mean


# 可视化同一张图像经过两次随机增强后的两个视图
view1, view2, label = contrastive_dataset[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(denormalize(view1, cifar10_mean, cifar10_std).permute(1, 2, 0).clamp(0, 1))
axes[0].set_title(f'view 1 / {classes[label]}')
axes[0].axis('off')

axes[1].imshow(denormalize(view2, cifar10_mean, cifar10_std).permute(1, 2, 0).clamp(0, 1))
axes[1].set_title(f'view 2 / {classes[label]}')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 3. 构建 DataLoader

In [ ]:
# 对比学习预训练用的 DataLoader
contrastive_loader = DataLoader(
    contrastive_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

# 线性评估阶段使用普通有标签数据
linear_train_loader = DataLoader(
    linear_train_dataset,
    batch_size=cfg.linear_batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

linear_test_loader = DataLoader(
    linear_test_dataset,
    batch_size=cfg.linear_batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

# 检查一个 batch 的形状
batch_view1, batch_view2, batch_labels = next(iter(contrastive_loader))
print('view1 shape:', batch_view1.shape)
print('view2 shape:', batch_view2.shape)
print('labels shape:', batch_labels.shape)

## 4. SimCLR 模型实现

SimCLR 一般由两部分组成：

1. `encoder`
   - 负责提取图像语义特征。

2. `projection head`
   - 把 encoder 输出映射到对比学习空间。
   - 训练时主要在这个空间上做对比损失。

实践中常见做法是：
- 预训练时使用 `projection head`
- 下游任务时更多使用 `encoder` 的特征

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, hidden_dim=512, out_dim=128):
        super().__init__()
        # 两层 MLP 是 SimCLR 里很常见的 projection head 设计
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)


class SimCLR(nn.Module):
    def __init__(self, projection_dim=128):
        super().__init__()
        # 使用 ResNet18 作为 backbone，去掉最后分类层
        backbone = models.resnet18(weights=None)
        feature_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.encoder = backbone
        self.projection_head = ProjectionHead(feature_dim, hidden_dim=512, out_dim=projection_dim)
        self.feature_dim = feature_dim
        self.projection_dim = projection_dim

    def forward(self, x):
        # h 是 encoder 输出的表征，z 是 projection head 输出的对比学习向量
        h = self.encoder(x)
        z = self.projection_head(h)
        # 对 z 做归一化，后面便于直接用余弦相似度
        z = F.normalize(z, dim=1)
        return h, z


model = SimCLR(projection_dim=cfg.projection_dim).to(device)
model

## 5. `NT-Xent / InfoNCE` Loss 详细解释

这是 SimCLR 最关键的部分。

设一个 batch 里有 `N` 张原图。每张图会产生两个增强视图，所以真正进入对比损失的是 `2N` 个样本。

### 5.1 正样本和负样本怎么定义

- 同一张原图产生的两个视图，互为正样本对。
- 一个 batch 中来自不同原图的其他视图，统统作为负样本。

例如：
- 第 1 张图产生 `x1_a` 和 `x1_b`
- 第 2 张图产生 `x2_a` 和 `x2_b`

那么：
- `x1_a` 的正样本是 `x1_b`
- `x1_a` 的负样本是 `x2_a, x2_b, ...`

### 5.2 相似度怎么计算

设 projection head 输出为 `z_i`，并且已经做过 L2 归一化。

这样两个向量的点积就等价于余弦相似度：

$$
sim(z_i, z_j) = z_i^T z_j
$$

### 5.3 单个样本的损失公式

对于样本 `i`，它的正样本是 `j`，那么损失写成：

$$
\ell_{i,j} = - \log \frac{\exp(sim(z_i, z_j) / \tau)}{\sum_{k=1}^{2N} \mathbf{1}_{[k \neq i]} \exp(sim(z_i, z_k) / \tau)}
$$

其中：
- `tau` 是温度系数
- 分子只放正样本 `j`
- 分母放除了自己 `i` 之外的所有样本，包括正样本和所有负样本

### 5.4 这是什么意思

这个公式在做的事情是：
- 希望 `i` 和正样本 `j` 的相似度尽可能大
- 同时希望 `i` 和其他负样本的相似度尽可能小

如果分子大、分母里负样本项小，那么整体 loss 就会变小。

### 5.5 为什么每对正样本要算两次

假设 `x1_a` 和 `x1_b` 是一对正样本：
- 会计算一次 `x1_a -> x1_b`
- 也会计算一次 `x1_b -> x1_a`

所以最终会在 `2N` 个样本上都各算一遍，再取平均。

In [ ]:
def nt_xent_loss(z1, z2, temperature=0.5):
    # z1 和 z2 的形状都是 [N, D]
    # 它们分别表示同一个 batch 中两次增强视图经过 projection head 后的表示
    batch_size = z1.size(0)

    # 把两组表示拼接起来，得到 [2N, D]
    # 前 N 个对应 view1，后 N 个对应 view2
    z = torch.cat([z1, z2], dim=0)

    # 由于 z 已经归一化过，这里直接矩阵乘法即可得到余弦相似度矩阵
    # similarity_matrix[i, j] 表示第 i 个样本和第 j 个样本的相似度
    similarity_matrix = torch.matmul(z, z.T)

    # 构造一个 mask，用来排除样本和它自己本身的相似度
    # 对角线位置是自己和自己，训练时不应该作为候选项参与分母
    self_mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z.device)

    # 构造正样本索引
    # 例如当 batch_size = N 时：
    # 0 的正样本是 N
    # 1 的正样本是 N+1
    # ...
    # N 的正样本是 0
    # N+1 的正样本是 1
    positive_indices = (torch.arange(2 * batch_size, device=z.device) + batch_size) % (2 * batch_size)

    # 先除以温度系数，温度越小，softmax 分布越尖锐
    logits = similarity_matrix / temperature

    # 为了避免自己和自己参与分母，把对角线位置填成一个极小值
    logits = logits.masked_fill(self_mask, -1e9)

    # 对每一行来说，正确类别就是它的正样本索引位置
    # 这就把对比学习转成了一个“2N 分类问题”：
    # 每个样本要在其余 2N-1 个候选里找出真正的正样本
    loss = F.cross_entropy(logits, positive_indices)

    return loss, similarity_matrix, logits, positive_indices


# 用一个小 batch 跑一遍，观察输出形状
batch_view1, batch_view2, _ = next(iter(contrastive_loader))
batch_view1 = batch_view1.to(device)
batch_view2 = batch_view2.to(device)

with torch.no_grad():
    _, z1 = model(batch_view1)
    _, z2 = model(batch_view2)
    loss, similarity_matrix, logits, positive_indices = nt_xent_loss(z1, z2, temperature=cfg.temperature)

print('z1 shape:', z1.shape)
print('z2 shape:', z2.shape)
print('similarity_matrix shape:', similarity_matrix.shape)
print('positive_indices shape:', positive_indices.shape)
print('loss:', float(loss))

## 6. 用更直观的话再解释一次 loss

假设一个 batch 里原本有 `N=4` 张图，那么增强后会变成 `2N=8` 个样本：

- `0,1,2,3` 来自第一组视图
- `4,5,6,7` 来自第二组视图

那么正样本配对关系是：

- `0 <-> 4`
- `1 <-> 5`
- `2 <-> 6`
- `3 <-> 7`

对于样本 `0` 来说：
- 分子：只取 `0` 和 `4` 的相似度
- 分母：取 `0` 和除自己之外所有样本的相似度，也就是 `1,2,3,4,5,6,7`

因此这个损失本质上是在问：

> 在所有候选样本里，模型能不能把真正配对的那个视图排到最前面？

如果能，loss 就小；如果不能，loss 就大。

## 7. 对比预训练函数

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=cfg.lr)


def train_one_epoch_contrastive(model, dataloader, optimizer, device, temperature):
    # 对比学习预训练模式
    model.train()
    running_loss = 0.0
    total = 0

    for view1, view2, _ in dataloader:
        view1 = view1.to(device)
        view2 = view2.to(device)

        optimizer.zero_grad()

        # 分别提取两组增强视图的表示
        _, z1 = model(view1)
        _, z2 = model(view2)

        # 使用 NT-Xent loss 让正样本更近、负样本更远
        loss, _, _, _ = nt_xent_loss(z1, z2, temperature=temperature)
        loss.backward()
        optimizer.step()

        batch_size = view1.size(0)
        running_loss += loss.item() * batch_size
        total += batch_size

    return running_loss / total

In [ ]:
# 对比预训练主循环
contrastive_history = []

for epoch in range(cfg.epochs):
    epoch_loss = train_one_epoch_contrastive(
        model=model,
        dataloader=contrastive_loader,
        optimizer=optimizer,
        device=device,
        temperature=cfg.temperature,
    )
    contrastive_history.append(epoch_loss)
    print(f'Epoch [{epoch + 1}/{cfg.epochs}] contrastive_loss={epoch_loss:.4f}')

In [ ]:
# 绘制对比学习损失曲线
epochs = range(1, len(contrastive_history) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, contrastive_history, marker='o')
plt.title('SimCLR contrastive loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

## 8. 提取 encoder 特征

SimCLR 预训练完成后，真正更有价值的通常是 `encoder` 学到的表示，而不是 projection head 本身。

下面把图像经过 encoder 后的特征提出来，后面可以接一个线性分类器做简单评估。

In [ ]:
@torch.no_grad()
def extract_features(model, dataloader, device):
    # 切换到评估模式，稳定提取特征
    model.eval()
    features = []
    labels = []

    for images, target in dataloader:
        images = images.to(device)
        # 这里取 encoder 的输出 h，而不是 projection head 的 z
        h = model.encoder(images)
        features.append(h.cpu())
        labels.append(target)

    features = torch.cat(features, dim=0)
    labels = torch.cat(labels, dim=0)
    return features, labels


train_features, train_labels = extract_features(model, linear_train_loader, device)
test_features, test_labels = extract_features(model, linear_test_loader, device)

print('train_features:', train_features.shape)
print('test_features:', test_features.shape)

## 9. 线性分类评估示例

线性评估的思路是：
- 冻结预训练好的 encoder
- 只在提取出的特征上训练一个线性分类器

如果线性分类效果不错，通常说明 encoder 学到的表示质量还可以。

In [ ]:
class LinearFeatureDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features.float()
        self.labels = labels.long()

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


linear_train_feature_loader = DataLoader(
    LinearFeatureDataset(train_features, train_labels),
    batch_size=cfg.linear_batch_size,
    shuffle=True,
)

linear_test_feature_loader = DataLoader(
    LinearFeatureDataset(test_features, test_labels),
    batch_size=cfg.linear_batch_size,
    shuffle=False,
)

# 线性分类头只做一个最简单的全连接分类
linear_classifier = nn.Linear(model.feature_dim, 10).to(device)
linear_optimizer = optim.Adam(linear_classifier.parameters(), lr=1e-3)
linear_criterion = nn.CrossEntropyLoss()

In [ ]:
def train_one_epoch_linear(classifier, dataloader, criterion, optimizer, device):
    classifier.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for features, labels in dataloader:
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = classifier(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * features.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total


@torch.no_grad()
def evaluate_linear(classifier, dataloader, criterion, device):
    classifier.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for features, labels in dataloader:
        features = features.to(device)
        labels = labels.to(device)

        logits = classifier(features)
        loss = criterion(logits, labels)

        running_loss += loss.item() * features.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total

In [ ]:
# 线性分类评估主循环
linear_history = {
    'train_loss': [],
    'train_acc': [],
    'test_loss': [],
    'test_acc': [],
}

for epoch in range(cfg.linear_epochs):
    train_loss, train_acc = train_one_epoch_linear(
        linear_classifier,
        linear_train_feature_loader,
        linear_criterion,
        linear_optimizer,
        device,
    )
    test_loss, test_acc = evaluate_linear(
        linear_classifier,
        linear_test_feature_loader,
        linear_criterion,
        device,
    )

    linear_history['train_loss'].append(train_loss)
    linear_history['train_acc'].append(train_acc)
    linear_history['test_loss'].append(test_loss)
    linear_history['test_acc'].append(test_acc)

    print(
        f'Epoch [{epoch + 1}/{cfg.linear_epochs}] '
        f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} '
        f'test_loss={test_loss:.4f} test_acc={test_acc:.4f}'
    )

## 10. SimCLR 和监督分类有什么不同？

1. 监督来源不同
   - 监督分类直接用标签。
   - SimCLR 用“同一张图的两个增强视图应该接近”作为监督信号。

2. 目标不同
   - 监督分类直接优化类别判别。
   - SimCLR 先优化表征空间，再把表征拿去做下游任务。

3. 损失函数不同
   - 监督分类一般用交叉熵做类别训练。
   - SimCLR 的 `NT-Xent` 本质上是在做“正样本匹配 + 负样本区分”。

## 11. 关键结论

- SimCLR 的关键不在类别标签，而在双视图构造。
- `NT-Xent loss` 的本质是：让正样本在 `2N-1` 个候选里尽可能排第一。
- 训练时主要使用 projection head 输出 `z` 做对比损失。
- 下游任务通常更关心 encoder 输出 `h` 的质量。